B20c. Организатор ВНЕ панели: снятие главного конфаунда.....


In [1]:
import sys, os, re, json
from itertools import combinations

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/panel_mechanisms')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
BASE = 'experiments/clinical_protocols/delphi_panels/'
GEM = BASE + 'heterogeneous_gemini_external_organizer/llm_as_judge/delphi_base_noroles_judge.csv'
REL = {'gpt (в панели)': PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv', 'qwen (в панели)': BASE + 'heterogeneous_qwen_organizer_complete/llm_as_judge/delphi_base_noroles_judge.csv', 'deepseekv4 (в панели)': BASE + 'heterogeneous_deepseek_organizer_complete/llm_as_judge/delphi_base_noroles_judge.csv', 'gemini (ВНЕ панели)': GEM}
D = {k: rd(v) for k, v in REL.items()}
A = load_arms(verbose=False)
M = meta()
ITEMS_ALL, POS = load_items(verbose=False)
SINGLE = {'gpt (в панели)': 0.6258, 'qwen (в панели)': 0.5622, 'deepseekv4 (в панели)': 0.6401, 'gemini (ВНЕ панели)': 0.4247}
print('плеч: %d, кейсов: %s' % (len(D), {k: len(v) for k, v in D.items()}))


плеч: 4, кейсов: {'gpt (в панели)': 229, 'qwen (в панели)': 229, 'deepseekv4 (в панели)': 229, 'gemini (ВНЕ панели)': 229}


In [2]:
rows = []
for k, d in D.items():
    ps = participants(d)
    wmu = d['winner_model_uri'].value_counts()
    cc = [c for c in d.columns if re.match('r[123]_completion_tokens__', c)]
    v = d[cc].apply(pd.to_numeric, errors='coerce')
    oc = [c for c in ['organizer_after_r1_completion_tokens', 'organizer_after_r2_completion_tokens', 'final_synthesis_completion_tokens'] if c in d.columns]
    ov = d[oc].apply(pd.to_numeric, errors='coerce')
    lab = pd.DataFrame({p: d['r1_emergency__' + p].map(norm_label) for p in ps})
    rows.append({'плечо': k, 'строк': len(d), 'модель протокола': list(wmu.index)[0].split('/')[-1] if len(wmu) else '?', 'единственная': len(wmu) == 1, 'участники': ', '.join((p.replace('_default', '') for p in ps)), 'на потолке (участн.)': int((v >= 10000).sum().sum()), 'на потолке (орг.)': int((ov >= 10000).sum().sum()), 'полнота голосов R1': float(lab.notna().all(axis=1).mean())})
CH = pd.DataFrame(rows).set_index('плечо')
show(CH, 'Целостность четырёх плеч')
print()
print('Состав участников во всех четырёх плечах ОДИНАКОВ. В плече gemini поле')
print('winner_model_uri равно gemini-3.7-flash во всех 229 кейсах, то есть')
print('протокол действительно писала модель, не входящая в панель.')



Целостность четырёх плеч
------------------------------------------------------------
                       строк  модель протокола  единственная                  участники  на потолке (участн.)  на потолке (орг.)  полнота голосов R1
плечо                                                                                                                                               
gpt (в панели)           229            latest          True  deepseekv4, gpt120b, qwen                     0                  0              0.9913
qwen (в панели)          229            latest          True  deepseekv4, gpt120b, qwen                     0                  0              1.0000
deepseekv4 (в панели)    229            latest          True  deepseekv4, gpt120b, qwen                     1                  3              0.9825
gemini (ВНЕ панели)      229  gemini-3.7-flash          True  deepseekv4, gpt120b, qwen                     2                  0              0.9913

Состав участников 

In [3]:
rows = []
for k, d in D.items():
    tk = [c for c in d.columns if re.match('r1_total_tokens__', c)]
    v = d[tk].apply(pd.to_numeric, errors='coerce')
    n_calls = int(v.notna().sum().sum())
    n_zero = int((v == 0).sum().sum())
    med = float(v[v > 0].stack().median())
    rows.append({'плечо': k, 'вызовов R1': n_calls, 'из кэша (0 токенов)': n_zero, 'доля кэша': n_zero / max(n_calls, 1), 'медиана токенов у реальных вызовов': med, 'записано R1 на кейс': float(v.sum(axis=1).mean())})
CA = pd.DataFrame(rows).set_index('плечо')
show(CA, 'Кэширование вызовов первого раунда')
print()
print('В плече gemini 38%% вызовов R1 отдано из кэша и записано с нулём токенов.')
print('Медиана токенов у РЕАЛЬНЫХ вызовов при этом совпадает с прочими плечами')
print('(≈2365), то есть сами ответы нормальные — занижен только УЧЁТ.')
print()
print('Проверим, не переиспользованы ли ответы из других плеч:')
ps = participants(D['gemini (ВНЕ панели)'])
for n, d in D.items():
    if 'gemini' in n:
        continue
    ix = D['gemini (ВНЕ панели)'].index.intersection(d.index)
    same = sum((int((D['gemini (ВНЕ панели)'].loc[ix, 'r1_raw__' + p].astype(str) == d.loc[ix, 'r1_raw__' + p].astype(str)).sum()) for p in ps))
    print('   совпадение текстов R1 с плечом %-22s %d из %d' % (n, same, len(ix) * len(ps)))
print()
print('Совпадений нет: прогон подлинно независимый, кэш сработал внутри самого')
print('прогона (перезапуск/докатка). Научная сопоставимость не нарушена,')
print('но БЮДЖЕТ этого плеча занижен и требует досчёта — см. шаг 8.')



Кэширование вызовов первого раунда
------------------------------------------------------------
                       вызовов R1  из кэша (0 токенов)  доля кэша  медиана токенов у реальных вызовов  записано R1 на кейс
плечо                                                                                                                     
gpt (в панели)                687                    0     0.0000                           2351.0000            7428.4978
qwen (в панели)               687                    0     0.0000                           2371.0000            7469.9825
deepseekv4 (в панели)         687                    0     0.0000                           2367.0000            7509.4847
gemini (ВНЕ панели)           687                  264     0.3843                           2365.0000            4620.8690

В плече gemini 38%% вызовов R1 отдано из кэша и записано с нулём токенов.
Медиана токенов у РЕАЛЬНЫХ вызовов при этом совпадает с прочими плечами
(≈2365), то есть с

In [4]:
S = pd.DataFrame({k: pd.to_numeric(v['Judge_score'], errors='coerce') for k, v in D.items()}).dropna()
res = pd.DataFrame({'Judge панели': S.mean(), 'SE': [se_clt(S[c]) for c in S.columns], 'организатор в одиночку': pd.Series(SINGLE)})
res['организатор в панели'] = ['да', 'да', 'да', 'НЕТ']
res = res.sort_values('Judge панели', ascending=False)
show(res, 'Четыре организатора, один и тот же состав участников')
print()
print('gemini — САМАЯ СЛАБАЯ модель пула (0.425 против 0.562-0.640) и БЕЗ ПРАВА')
print('ГОЛОСА — даёт второй результат из четырёх.')



Четыре организатора, один и тот же состав участников
------------------------------------------------------------
                       Judge панели     SE  организатор в одиночку организатор в панели
deepseekv4 (в панели)        0.8111 0.0081                  0.6401                   да
gemini (ВНЕ панели)          0.7995 0.0093                  0.4247                  НЕТ
gpt (в панели)               0.7899 0.0084                  0.6258                   да
qwen (в панели)              0.7592 0.0093                  0.5622                   да

gemini — САМАЯ СЛАБАЯ модель пула (0.425 против 0.562-0.640) и БЕЗ ПРАВА
ГОЛОСА — даёт второй результат из четырёх.


In [5]:
rows = []
for a, b in combinations(S.columns, 2):
    d = S[a] - S[b]
    ct = cluster_test(d, M['Sub Clinical Specialties'].reindex(d.index))
    g = M['Sub Clinical Specialties'].reindex(d.index)
    gm = d.groupby(g).mean()
    se_dom = float(np.sqrt(se_clt(d) ** 2 + gm.var(ddof=1) / gm.size))
    rows.append({'контраст': '%s − %s' % (a, b), 'Δ': float(d.mean()), 'SE кластер': ct['se_cluster'], 'SE + домен': se_dom, 'p Уилкоксон': float(stats.wilcoxon(d)[1]), 'p кластер': ct['p']})
C = pd.DataFrame(rows)
C['p Холм (6 пар)'] = holm(C['p Уилкоксон'])
C['вердикт'] = np.where(C['p Холм (6 пар)'] < 0.05, 'значим', 'НЕ значим')
show(C.set_index('контраст'), 'Все шесть попарных контрастов')
print()
print('КЛЮЧЕВОЕ: gemini статистически НЕОТЛИЧИМ и от лучшего организатора')
print('(deepseekv4), и от gpt, но ЗНАЧИМО ЛУЧШЕ qwen.')



Все шесть попарных контрастов
------------------------------------------------------------
                                                  Δ  SE кластер  SE + домен  p Уилкоксон  p кластер  p Холм (6 пар)    вердикт
контраст                                                                                                                      
gpt (в панели) − qwen (в панели)             0.0308      0.0090      0.0116       0.0001     0.0030          0.0005     значим
gpt (в панели) − deepseekv4 (в панели)      -0.0211      0.0077      0.0141       0.0133     0.0134          0.0398     значим
gpt (в панели) − gemini (ВНЕ панели)        -0.0096      0.0099      0.0126       0.1402     0.3462          0.2804  НЕ значим
qwen (в панели) − deepseekv4 (в панели)     -0.0519      0.0066      0.0110       0.0000     0.0000          0.0000     значим
qwen (в панели) − gemini (ВНЕ панели)       -0.0403      0.0087      0.0115       0.0000     0.0002          0.0000     значим
deepseekv4 (в панел

In [6]:
rk = res[['Judge панели', 'организатор в одиночку']].copy()
rk['ранг по панели'] = rk['Judge панели'].rank(ascending=False)
rk['ранг по силе'] = rk['организатор в одиночку'].rank(ascending=False)
rk['сдвиг ранга'] = rk['ранг по панели'] - rk['ранг по силе']
show(rk, 'Ранги: качество панели против силы организатора')
rho4, p4 = stats.spearmanr(rk['Judge панели'], rk['организатор в одиночку'])
sub3 = rk[rk.index != 'gemini (ВНЕ панели)']
rho3, _ = stats.spearmanr(sub3['Judge панели'], sub3['организатор в одиночку'])
print()
print('ранговая корреляция при n = 3 (только организаторы в панели): rho = %+.3f' % rho3)
print('ранговая корреляция при n = 4 (с gemini вне панели)         : rho = %+.3f' % rho4)
print()
print('Связь резко ослабевает. Слабейшая модель пула поднимается на второе')
print('место, обгоняя два более сильных организатора. Версия «качество панели')
print('равно качеству личного ответа организатора» не подтверждается.')



Ранги: качество панели против силы организатора
------------------------------------------------------------
                       Judge панели  организатор в одиночку  ранг по панели  ранг по силе  сдвиг ранга
deepseekv4 (в панели)        0.8111                  0.6401          1.0000        1.0000       0.0000
gemini (ВНЕ панели)          0.7995                  0.4247          2.0000        4.0000      -2.0000
gpt (в панели)               0.7899                  0.6258          3.0000        2.0000       1.0000
qwen (в панели)              0.7592                  0.5622          4.0000        3.0000       1.0000

ранговая корреляция при n = 3 (только организаторы в панели): rho = +1.000
ранговая корреляция при n = 4 (с gemini вне панели)         : rho = +0.400

Связь резко ослабевает. Слабейшая модель пула поднимается на второе
место, обгоняя два более сильных организатора. Версия «качество панели
равно качеству личного ответа организатора» не подтверждается.


In [7]:
ITEMS = {k: item_frame(v) for k, v in D.items()}
common = set(POS)
for f in ITEMS.values():
    common &= set(f)
KEYS = sorted(common)
MEMB = ['baseline gpt', 'baseline qwen', 'baseline deepseekv4']
pts = {k: ITEMS_ALL[MEMB[0]][k]['points'] for k in KEYS}
maj = {k: int(sum((int(ITEMS_ALL[m][k]['met']) for m in MEMB)) >= 2) for k in KEYS}
uni = {k: int(sum((int(ITEMS_ALL[m][k]['met']) for m in MEMB)) >= 1) for k in KEYS}
print('общих пунктов во всех четырёх плечах: %d' % len(KEYS))

def per_case(fn):
    d = pd.DataFrame([{'case': k[0], 'v': fn(k), 'p': pts[k]} for k in KEYS])
    return d.groupby('case').apply(lambda g: (g['v'] * g['p']).sum() / g['p'].sum(), include_groups=False)
ms, us = (per_case(lambda k: maj[k]), per_case(lambda k: uni[k]))
rows, lift = ([], {})
for org, f in ITEMS.items():
    p_ = per_case(lambda k, f=f: int(f[k]['met']))
    ix = p_.index.intersection(ms.index)
    lift[org] = p_.loc[ix] - ms.loc[ix]
    em = sum((1 for k in KEYS if int(f[k]['met']) == 1 and sum((int(ITEMS_ALL[m][k]['met']) for m in MEMB)) == 0))
    lo = sum((1 for k in KEYS if int(f[k]['met']) == 0 and sum((int(ITEMS_ALL[m][k]['met']) for m in MEMB)) >= 2))
    rows.append({'организатор': org, 'панель': float(p_.loc[ix].mean()), 'большинство': float(ms.loc[ix].mean()), 'объединение (оракул)': float(us.loc[ix].mean()), 'вклад синтеза': float(lift[org].mean()), 'превышение оракула': float((p_.loc[ix] - us.loc[ix]).mean()), 'эмерджентных': em, 'потерянных': lo, 'нетто': em - lo})
LIFT = pd.DataFrame(rows).set_index('организатор')
show(LIFT, 'Вклад организатора и попунктный баланс')
print()
print('gemini даёт ВТОРОЙ по величине вклад синтеза и НАИМЕНЬШЕЕ число')
print('потерянных пунктов из всех четырёх. При этом собственного голоса у него')
print('нет: он сохраняет чужие находки лучше, чем организаторы-участники.')


общих пунктов во всех четырёх плечах: 2249



Вклад организатора и попунктный баланс
------------------------------------------------------------
                       панель  большинство  объединение (оракул)  вклад синтеза  превышение оракула  эмерджентных  потерянных  нетто
организатор                                                                                                                         
gpt (в панели)         0.7900       0.6295                0.7688         0.1605              0.0211           246          69    177
qwen (в панели)        0.7620       0.6295                0.7688         0.1325             -0.0069           214          85    129
deepseekv4 (в панели)  0.8141       0.6295                0.7688         0.1846              0.0452           275          57    218
gemini (ВНЕ панели)    0.8033       0.6295                0.7688         0.1739              0.0345           248          54    194

gemini даёт ВТОРОЙ по величине вклад синтеза и НАИМЕНЬШЕЕ число
потерянных пунктов из всех четырёх. 

In [8]:
rows = []
for org, f in ITEMS.items():
    r = {'организатор': org}
    for m in MEMB:
        mm = m.replace('baseline ', '')
        r['совпадение с ' + mm] = float(np.mean([int(f[k]['met']) == int(ITEMS_ALL[m][k]['met']) for k in KEYS]))
    r['ближайший участник'] = max(MEMB, key=lambda m: np.mean([int(f[k]['met']) == int(ITEMS_ALL[m][k]['met']) for k in KEYS])).replace('baseline ', '')
    rows.append(r)
DOM = pd.DataFrame(rows).set_index('организатор')
show(DOM, 'С чьими одиночными вердиктами совпадает итоговый протокол')
print()
print('Во всех четырёх плечах ближайшим оказывается deepseekv4 — сильнейший')
print('участник. Организатор своим мнением протокол не перетягивает: в плече')
print('gemini мнения у него нет вовсе, а картина та же.')
print()
print('ВЫВОД: версия (б) «самопредпочтение» ОТВЕРГАЕТСЯ. Работает версия (а):')
print('организатор занимается синтезом, а содержание тянется к сильнейшему')
print('участнику панели.')



С чьими одиночными вердиктами совпадает итоговый протокол
------------------------------------------------------------
                       совпадение с gpt  совпадение с qwen  совпадение с deepseekv4 ближайший участник
организатор                                                                                           
gpt (в панели)                   0.7430             0.7065                   0.7590         deepseekv4
qwen (в панели)                  0.7479             0.7381                   0.7675         deepseekv4
deepseekv4 (в панели)            0.7261             0.6932                   0.7519         deepseekv4
gemini (ВНЕ панели)              0.7421             0.7145                   0.7563         deepseekv4

Во всех четырёх плечах ближайшим оказывается deepseekv4 — сильнейший
участник. Организатор своим мнением протокол не перетягивает: в плече
gemini мнения у него нет вовсе, а картина та же.

ВЫВОД: версия (б) «самопредпочтение» ОТВЕРГАЕТСЯ. Работает версия (а):
о

In [9]:
AX = ['accuracy', 'completeness', 'reasoning', 'urgency', 'uncertainty', 'communication_quality']
axis_of = {k: ITEMS_ALL[MEMB[0]][k]['axis'] for k in KEYS}
W = pd.Series({a: sum((pts[k] for k in KEYS if axis_of[k] == a)) for a in AX})
W = W / W.sum()

def axis_scores(f):
    out = {}
    for a in AX:
        ks = [k for k in KEYS if axis_of[k] == a]
        out[a] = sum((int(f[k]['met']) * pts[k] for k in ks)) / sum((pts[k] for k in ks)) if ks else np.nan
    return pd.Series(out)
AS = pd.DataFrame({o: axis_scores(f) for o, f in ITEMS.items()}).T
show(AS[AX], 'Доля выполненных пунктов по осям')
dec = pd.DataFrame({'вес оси': W, 'gemini − qwen': AS.loc['gemini (ВНЕ панели)'] - AS.loc['qwen (в панели)']})
dec['вклад'] = dec['gemini − qwen'] * dec['вес оси']
dec['доля вклада, %'] = 100 * dec['вклад'] / dec['вклад'].sum()
show(dec.loc[AX].sort_values('вклад', ascending=False), 'Разложение преимущества gemini над qwen по осям')
print()
print('Ориентиры: в C28 разрыв «панель против одиночной модели» на 42.9%% давала')
print('completeness и на 19.3%% uncertainty; в B20b разрыв между организаторами —')
print('43.8%% и 19.6%%. Сравните с профилем выше.')



Доля выполненных пунктов по осям
------------------------------------------------------------
                       accuracy  completeness  reasoning  urgency  uncertainty  communication_quality
gpt (в панели)           0.8903        0.7228     0.8991   0.8963       0.5567                 0.6429
qwen (в панели)          0.8925        0.6408     0.8618   0.8893       0.5671                 0.6490
deepseekv4 (в панели)    0.9027        0.7134     0.9352   0.9405       0.6729                 0.6786
gemini (ВНЕ панели)      0.9115        0.6950     0.9144   0.9416       0.6383                 0.6296

Разложение преимущества gemini над qwen по осям
------------------------------------------------------------
                       вес оси  gemini − qwen   вклад  доля вклада, %
completeness            0.3201         0.0541  0.0173         42.5710
uncertainty             0.0983         0.0712  0.0070         17.1953
urgency                 0.1246         0.0524  0.0065         16.0267
reaso

In [10]:
parts = []
for org, d in D.items():
    s = pd.to_numeric(d['Judge_score'], errors='coerce')
    w = d['winner_protocol'].map(words)
    ix = s.dropna().index.intersection(w.dropna().index)
    parts.append(pd.DataFrame({'y': s.loc[ix].to_numpy(), 'w': w.loc[ix].to_numpy(), 'org': org, 'case': ix}))
L = pd.concat(parts, ignore_index=True)
L['lw'] = np.log(L['w'].clip(lower=1))
show(L.groupby('org').agg(медиана_слов=('w', 'median'), среднее=('w', 'mean')), 'Длина итогового протокола')
Dm = pd.get_dummies(L['org'], prefix='o', drop_first=True).astype(float)
g = L['case'].astype('category').cat.codes
m0 = sm.OLS(L['y'], sm.add_constant(Dm)).fit(cov_type='cluster', cov_kwds={'groups': g})
m1 = sm.OLS(L['y'], sm.add_constant(pd.concat([Dm, L[['lw']]], axis=1))).fit(cov_type='cluster', cov_kwds={'groups': g})
show(pd.DataFrame({'без контроля длины': m0.params, 'p': m0.pvalues, 'с контролем длины': m1.params, 'p ': m1.pvalues}), 'Эффект организатора до и после контроля длины (база — deepseekv4)')
print()
print('После контроля длины НИ ОДИН контраст организаторов не значим. Длина')
print('остаётся сильным предиктором. Ограничение то же, что в B24 и B20b, и')
print('закрывается только каузальным тестом длины.')
print()
print('ЧТО ЗДЕСЬ ВСЁ ЖЕ НОВОГО. gemini достигает результата уровня deepseekv4,')
print('написав на ~100 слов меньше. При контроле длины его коэффициент')
print('положителен относительно deepseekv4, то есть при равной длине он не')
print('уступает. Это довод в пользу того, что дело не только в многословии.')



Длина итогового протокола
------------------------------------------------------------
                       медиана_слов  среднее
org                                         
deepseekv4 (в панели)      608.0000 604.7555
gemini (ВНЕ панели)        506.0000 530.3930
gpt (в панели)             472.0000 478.5371
qwen (в панели)            464.0000 468.4236

Эффект организатора до и после контроля длины (база — deepseekv4)
-----------------------------------------------------------------
                       без контроля длины      p  с контролем длины     p 
const                              0.8111 0.0000            -0.0796 0.7096
lw                                    NaN    NaN             0.1393 0.0000
o_gemini (ВНЕ панели)             -0.0115 0.2042             0.0080 0.4019
o_gpt (в панели)                  -0.0211 0.0066             0.0119 0.2629
o_qwen (в панели)                 -0.0519 0.0000            -0.0161 0.1443

После контроля длины НИ ОДИН контраст организаторов не зна

In [11]:
rows = []
for org, d in D.items():
    t = true_label(d).map(ORD)
    p = pred_triage(d).map(ORD)
    ok = t.notna() & p.notna()
    tk1 = [c for c in d.columns if re.match('r1_total_tokens__', c)]
    v1 = d[tk1].apply(pd.to_numeric, errors='coerce')
    n_zero = int((v1 == 0).sum().sum())
    med = float(v1[v1 > 0].stack().median())
    r1_rep = float(v1.sum(axis=1).mean())
    r1_true = r1_rep + n_zero / len(d) * med
    r23 = sum((d[[c for c in d.columns if re.match('r%d_total_tokens__' % r, c)]].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1).mean() for r in (2, 3)))
    oc = [c for c in ['organizer_after_r1_total_tokens', 'organizer_after_r2_total_tokens', 'final_synthesis_total_tokens'] if c in d.columns]
    agg = d[oc].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1).mean()
    jm = float(pd.to_numeric(d['Judge_score'], errors='coerce').mean())
    rows.append({'плечо': org, 'accuracy триажа': float((p[ok] == t[ok]).mean()), 'недотриаж': float((p[ok] < t[ok]).mean()), 'перетриаж': float((p[ok] > t[ok]).mean()), 'S записано': r1_rep + r23, 'S с досчётом': r1_true + r23, 'V агрегация': agg, 'total записано': r1_rep + r23 + agg, 'total с досчётом': r1_true + r23 + agg, 'Judge/10k (досчёт)': jm / ((r1_true + r23 + agg) / 10000.0)})
B = pd.DataFrame(rows).set_index('плечо')
show(B, 'Триаж и бюджет')
print()
print('ИСПРАВЛЕНИЕ. По записанным токенам плечо gemini выглядело самым дешёвым')
print('и самым эффективным. После досчёта кэшированных вызовов оно оказывается')
print('сопоставимым с остальными. Заявлять экономию по этому плечу НЕЛЬЗЯ.')



Триаж и бюджет
------------------------------------------------------------
                       accuracy триажа  недотриаж  перетриаж  S записано  S с досчётом  V агрегация  total записано  total с досчётом  Judge/10k (досчёт)
плечо                                                                                                                                                    
gpt (в панели)                  0.7860     0.0131     0.2009  14537.0131    14537.0131   12209.4410      26746.4541        26746.4541              0.2953
qwen (в панели)                 0.8297     0.0044     0.1659  14586.4367    14586.4367   10925.9782      25512.4148        25512.4148              0.2976
deepseekv4 (в панели)           0.8253     0.0087     0.1659  14508.5415    14508.5415   14574.3319      29082.8734        29082.8734              0.2789
gemini (ВНЕ панели)             0.8166     0.0087     0.1747  10448.8122    13175.2751   14535.8297      24984.6419        27711.1048              0.2885

In [12]:
rows = []
for org, d in D.items():
    te = ts = 0
    for ce, cs in zip(d['delphi_emergency_counts_json'].fillna('{}'), d['delphi_subclinical_counts_json'].fillna('{}')):
        try:
            e, s_ = (json.loads(ce), json.loads(cs))
        except Exception:
            continue
        if e and sum((1 for v in e.values() if v == max(e.values()))) > 1:
            te += 1
        if s_ and sum((1 for v in s_.values() if v == max(s_.values()))) > 1:
            ts += 1
    rows.append({'плечо': org, 'ничья по срочности': te, 'ничья по специальности': ts, 'доля кейсов с ничьёй': (te + ts) / len(d)})
show(pd.DataFrame(rows).set_index('плечо'), 'Как часто вообще возникала ничья')
print()
print('Ничьи единичны, поэтому подмена правила не могла повлиять на сравнение.')



Как часто вообще возникала ничья
------------------------------------------------------------
                       ничья по срочности  ничья по специальности  доля кейсов с ничьёй
плечо                                                                                  
gpt (в панели)                          0                       1                0.0044
qwen (в панели)                         0                       0                0.0000
deepseekv4 (в панели)                   0                       1                0.0044
gemini (ВНЕ панели)                     0                       2                0.0087

Ничьи единичны, поэтому подмена правила не могла повлиять на сравнение.


In [13]:
save(CH.reset_index(), 'organizer_outside_panel_integrity.csv')
save(CA.reset_index(), 'organizer_outside_panel_cache.csv')
save(res.reset_index(), 'organizer_outside_panel_headline.csv')
save(C, 'organizer_outside_panel_contrasts.csv')
save(LIFT.reset_index(), 'organizer_outside_panel_lift.csv')
save(DOM.reset_index(), 'organizer_outside_panel_dominance.csv')
save(AS.reset_index(), 'organizer_outside_panel_axes.csv')
save(B.reset_index(), 'organizer_outside_panel_budget.csv')


сохранено: out/organizer_outside_panel_integrity.csv  (4 строк)
сохранено: out/organizer_outside_panel_cache.csv  (4 строк)
сохранено: out/organizer_outside_panel_headline.csv  (4 строк)
сохранено: out/organizer_outside_panel_contrasts.csv  (6 строк)
сохранено: out/organizer_outside_panel_lift.csv  (4 строк)
сохранено: out/organizer_outside_panel_dominance.csv  (4 строк)
сохранено: out/organizer_outside_panel_axes.csv  (4 строк)
сохранено: out/organizer_outside_panel_budget.csv  (4 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\panel_mechanisms\\results\\organizer_outside_panel_budget.csv'